In [10]:
# This is an example file showing how to train a model
import os
import torch
import albumentations as A
import numpy as np
import glob
import rioxarray

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from treecrowndelineation import TreeCrownDelineationModel
from treecrowndelineation.dataloading.in_memory_datamodule import InMemoryDataModule


In [11]:
###################################
#      file paths and settings    #
###################################
rasters = np.sort(glob.glob('./training/data/tiles/*.tif'))
masks   = np.sort(glob.glob('./training/data/masks/*.tif'))
outlines = np.sort(glob.glob('./training/data/outlines/*.tif'))
dist     = np.sort(glob.glob('./training/data/dist_trafo/*.tif'))

print(len(rasters), len(masks), len(outlines), len(dist))  # 数量应该一致

rasters_3band = [rioxarray.open_rasterio(r).values[:3] for r in rasters]
print(len(rasters_3band))
print(rasters_3band[0].shape)  # 看第一个瓦片的形状



logdir = "./out/logs/"
model_save_path = "./out/models/"
experiment_name = "Block21"

arch = "Unet-resnet18"
width = 256
batchsize = 16
in_channels = 3 # number of input channels, e.g. 3 for RGB, 4 for RGBI
devices = 1  # number of gpus, if you have multiple
accelerator = "auto"  # or gpu or cpu, see lightning docs
max_epochs = 30 + 60 - 1
lr = 3E-4

training_split = 0.8

model_name = "{}_epochs={}_lr={}_width={}_bs={}".format(arch,
                                                        max_epochs,
                                                        lr,
                                                        width,
                                                        batchsize)






37 37 37 37
37
(3, 174, 199)


In [12]:
#%%
###################################
#             training            #
###################################
logger = TensorBoardLogger(logdir,
                           name=experiment_name,
                           version=model_name,
                           default_hp_metric=False)

cp = ModelCheckpoint(os.path.abspath(model_save_path) + "/" + experiment_name,
                     model_name + "-{epoch}",
                     monitor="val/loss",
                     save_last=True,
                     save_top_k=2)

callbacks = [cp, LearningRateMonitor()]

train_augmentation = A.Compose([A.RandomCrop(width, width),
                                A.RandomRotate90(),
                                A.VerticalFlip()
                                ])
val_augmentation = A.RandomCrop(width, width)



data = InMemoryDataModule(rasters_3band,
                          (masks, outlines, dist),
                          width=width,
                          batchsize=batchsize,
                          training_split=training_split,
                          train_augmentation=train_augmentation,
                          val_augmentation=val_augmentation,
                          concatenate_ndvi=False, 
                          dilate_second_target_band=0,
                          rescale_ndvi=False)



model = TreeCrownDelineationModel(in_channels=in_channels, lr=lr)

#%%
trainer = Trainer(devices=devices,
                  accelerator=accelerator,
                  logger=logger,  
                  callbacks=callbacks,
                  # checkpoint_callback=False,  # set this to avoid logging into the working directory
                  max_epochs=max_epochs)
trainer.fit(model, data)

#%%
model.to("cpu")
t = torch.rand(1, in_channels, width, width, dtype=torch.float32)
model.to_torchscript(
    os.path.abspath(model_save_path) + "/" + experiment_name + '/' + model_name + "_jitted.pt",
    method="trace",
    example_inputs=t)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\Patrick\miniconda3\envs\geo_311\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory C:\Users\Patrick\Documents\GitHub\TreeCrownDelineation\run\out\models\Block21 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ seg_model  │ SegmentationModel │ 14.3 M │ train │     0 │
│ 1 │ dist_model │ DistanceModel     │  890 K │ train │     0 │
└───┴────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 15.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 15.2 M                                                                                               
Total estimated model params size (MB): 60                                                                         
Modules in train mode: 226                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Patrick\miniconda3\envs\geo_311\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

c:\Users\Patrick\miniconda3\envs\geo_311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:4
34: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.

c:\Users\Patrick\miniconda3\envs\geo_311\Lib\site-packages\pytorch_lightning\utilities\data.py:106: Total length of
`CombinedLoader` across ranks is zero. Please make sure this was your intention.

`Trainer.fit` stopped: No training batches.


c:\Users\Patrick\miniconda3\envs\geo_311\Lib\site-packages\pytorch_lightning\core\module.py:1550: `LightningModule.to_torchscript` has been deprecated in v2.7 and will be removed in v2.8. TorchScript is deprecated in PyTorch. Use `torch.export.export()` for model exporting instead. See https://pytorch.org/docs/stable/export.html for more information.


TreeCrownDelineationModel(
  original_name=TreeCrownDelineationModel
  (seg_model): SegmentationModel(
    original_name=SegmentationModel
    (model): Unet(
      original_name=Unet
      (encoder): ResNetEncoder(
        original_name=ResNetEncoder
        (conv1): Conv2d(original_name=Conv2d)
        (bn1): BatchNorm2d(original_name=BatchNorm2d)
        (relu): ReLU(original_name=ReLU)
        (maxpool): MaxPool2d(original_name=MaxPool2d)
        (layer1): Sequential(
          original_name=Sequential
          (0): BasicBlock(
            original_name=BasicBlock
            (conv1): Conv2d(original_name=Conv2d)
            (bn1): BatchNorm2d(original_name=BatchNorm2d)
            (relu): ReLU(original_name=ReLU)
            (conv2): Conv2d(original_name=Conv2d)
            (bn2): BatchNorm2d(original_name=BatchNorm2d)
          )
          (1): BasicBlock(
            original_name=BasicBlock
            (conv1): Conv2d(original_name=Conv2d)
            (bn1): BatchNorm2d(origina

In [13]:
import torch
import numpy as np
import rioxarray

# 1. Using pre-trained models

model = torch.jit.load(r'C:\Users\Patrick\Documents\GitHub\TreeCrownDelineation\run\out\models\Block21\Unet-resnet18_epochs=89_lr=0.0003_width=256_bs=16_jitted.pt')
model = model.to('cuda')
model.eval()

# 2. Load image
img = rioxarray.open_rasterio('./training/K2-21_GDA94_MGA52.tif', lock=False).values.astype(np.float32) / 255.0
img = img[:3]  # 只取RGB
tensor = torch.from_numpy(img).unsqueeze(0).float()
print(tensor.shape)  # 确认是 (1, 3, H, W)

torch.Size([1, 3, 10606, 9991])


In [14]:
def pad_to_multiple(tensor, multiple=32):
    _, _, h, w = tensor.shape
    new_h = ((h + multiple - 1) // multiple) * multiple
    new_w = ((w + multiple - 1) // multiple) * multiple
    pad_h = new_h - h
    pad_w = new_w - w
    return torch.nn.functional.pad(tensor, (0, pad_w, 0, pad_h)), h, w

def predict_in_tiles(model, tensor, tile_size=1024, overlap=32, device='cuda'):
    _, c, h, w = tensor.shape
    with torch.no_grad():
        test = model(tensor[:, :, :32, :32].to(device))
    out_channels = test.shape[1]
    output = torch.zeros(1, out_channels, h, w)
    for y in range(0, h, tile_size - overlap):
        for x in range(0, w, tile_size - overlap):
            y2 = min(y + tile_size, h)
            x2 = min(x + tile_size, w)
            tile = tensor[:, :, y:y2, x:x2]
            tile_padded, th, tw = pad_to_multiple(tile)
            with torch.no_grad():
                pred = model(tile_padded.to(device))
            output[:, :, y:y2, x:x2] = pred[:, :, :th, :tw].cpu()
            torch.cuda.empty_cache()
            print(f"进度: y={y}/{h}, x={x}/{w}")
    return output

output = predict_in_tiles(model, tensor, tile_size=1024, device='cuda')
print(f"输出尺寸: {output.shape}")

进度: y=0/10606, x=0/9991
进度: y=0/10606, x=992/9991
进度: y=0/10606, x=1984/9991
进度: y=0/10606, x=2976/9991
进度: y=0/10606, x=3968/9991
进度: y=0/10606, x=4960/9991
进度: y=0/10606, x=5952/9991
进度: y=0/10606, x=6944/9991
进度: y=0/10606, x=7936/9991
进度: y=0/10606, x=8928/9991
进度: y=0/10606, x=9920/9991
进度: y=992/10606, x=0/9991
进度: y=992/10606, x=992/9991
进度: y=992/10606, x=1984/9991
进度: y=992/10606, x=2976/9991
进度: y=992/10606, x=3968/9991
进度: y=992/10606, x=4960/9991
进度: y=992/10606, x=5952/9991
进度: y=992/10606, x=6944/9991
进度: y=992/10606, x=7936/9991
进度: y=992/10606, x=8928/9991
进度: y=992/10606, x=9920/9991
进度: y=1984/10606, x=0/9991
进度: y=1984/10606, x=992/9991
进度: y=1984/10606, x=1984/9991
进度: y=1984/10606, x=2976/9991
进度: y=1984/10606, x=3968/9991
进度: y=1984/10606, x=4960/9991
进度: y=1984/10606, x=5952/9991
进度: y=1984/10606, x=6944/9991
进度: y=1984/10606, x=7936/9991
进度: y=1984/10606, x=8928/9991
进度: y=1984/10606, x=9920/9991
进度: y=2976/10606, x=0/9991
进度: y=2976/10606, x=992/9991
进度: y=2976

In [15]:
import rioxarray
import xarray as xr
import numpy as np

# 加载原始图像获取地理信息
src = rioxarray.open_rasterio('./training/K2-21_GDA94_MGA52.tif')

# 转换输出为numpy
out_arr = output.squeeze(0).numpy()  # (3, H, W)

# 创建xarray并附上地理信息
out_xr = xr.DataArray(
    out_arr,
    dims=['band', 'y', 'x'],
    coords={'band': [1, 2, 3], 'y': src.y, 'x': src.x}
)
out_xr = out_xr.rio.write_crs(src.rio.crs)
out_xr = out_xr.rio.write_transform(src.rio.transform())

# 保存
out_xr.rio.to_raster('./out/output_prediction.tif')
print("保存完成！")

保存完成！


In [17]:
print(out_xr.min(), out_xr.max())

<xarray.DataArray ()> Size: 4B
array(-6.105531, dtype=float32)
Coordinates:
    spatial_ref  int32 4B 0 <xarray.DataArray ()> Size: 4B
array(0.8076106, dtype=float32)
Coordinates:
    spatial_ref  int32 4B 0


In [19]:
print("input range:", out_xr.min(), out_xr.max())


input range: <xarray.DataArray ()> Size: 4B
array(-6.105531, dtype=float32)
Coordinates:
    spatial_ref  int32 4B 0 <xarray.DataArray ()> Size: 4B
array(0.8076106, dtype=float32)
Coordinates:
    spatial_ref  int32 4B 0
